# Stage 2: data audits

Backgrounds (2a), near-duplicates (2b), overlap with Paddy Doctor (2c) and a label sanity check (2d)
(handbook Stage 2). All logic lives in `src/duplicates.py`, `src/background_audit.py`, `src/label_check.py`
and `src/audit_report.py`; this notebook only calls them. The human parts (labelling, reviewing, checking)
run in a terminal; re-run this notebook afterwards and the report fills in.


In [ ]:
from IPython.display import Markdown, display

from src import background_audit, duplicates
from src.audit_report import write_report
from src.config import load_config

config = load_config("configs/audits.yaml")

## 1. Near-duplicate candidates (2b, 2c)

Hashes every photo of both datasets in 8 orientations, describes every whole photo with E1 (GPU), flags pairs by
the two pre-declared rules, runs the automatic keypoint check and writes the blind review queue. The caches are
git-ignored and reused. Refuses to run again once anyone has started reviewing (the queue is frozen).


In [ ]:
from src.config import REPO_ROOT

if not (REPO_ROOT / config["duplicates"]["review_file"]).exists():
    duplicates.flag(config)

## 2. Background sample (2a)

60 random photos per class, drawn once with a fixed seed and frozen.


In [ ]:
background_audit.sample(config)["class"].value_counts()

## 3. People, in a terminal

Each runs alone, without looking at anyone else's answers:

```
uv run python -m src.background_audit label --rater <name>     # two team members
uv run python -m src.duplicates review --reviewer <name>       # two team members
uv run python -m src.label_check check --checker <name>        # every team member
```

Then, together: `uv run python -m src.background_audit settle` and `uv run python -m src.duplicates settle`.
Finally `uv run python -m src.background_audit results`, `uv run python -m src.duplicates combine` and
`uv run python -m src.label_check results` write the final files.


## 4. Report

In [ ]:
display(Markdown(write_report(config)))